# GameTheory-12 — Jeux de Réputation (twin C# du notebook Python)

Ce notebook est le **jumeau C# / .NET 9** de `GameTheory-12-ReputationGames.ipynb` (Python + numpy + matplotlib).
Il implémente from-scratch (BCL .NET 9, **0 NuGet**) les mêmes algorithmes de **théorie des jeux de réputation** :

- **Paradoxe de la chaîne de magasins** (Selten 1978) — induction arrière en information complète.
- **Cheap talk Crawford-Sobel (1982)** — équilibres de partition, effet du biais sur l'information transmise.
- **Kreps-Wilson (1982)** — chaîne de magasins avec une petite probabilité $\epsilon$ d'un incumbent irrationnel ("tough") ;
  mise à jour bayésienne des croyances -> la réputation devient crédible.
- **KMRW (Kreps-Milgrom-Roberts-Wilson 1982)** — dilemme du prisonnier fini : une petite incertitude sur le type
  (Tit-for-Tat vs rationnel) permet la coopération.
- **Équilibre bayésien parfait (PBE)** — stratégies + système de croyances.

**Filiation d'algorithmes** : ce notebook est distinct de `GameTheory-4-NashEquilibrium` (équilibre en information complète),
`GameTheory-13-ImperfectInfo-CFR` (regret counterfactuel dans le jeu répliqué) et `GameTheory-16-MechanismDesign`
(conception de mecanismes). Ici le coeur est la **mise à jour bayésienne des croyances** et la **dynamique de réputation**.

> **Note de parité (Prong B, EPIC #3801)** : le notebook Python s'appuie sur numpy (PRNG Mersenne Twister) pour le
> tirage Monte-Carlo. Le twin C# utilise `System.Random` (seed fixe pour reproductibilité) ; les sorties analytiques
> (Crawford-Sobel, Kreps-Wilson, KMRW) sont exactement identiques, les sorties Monte-Carlo sont qualitativement équivalentes
> (variabilité PRNG attendue). matplotlib -> tables console / ASCII (convention GT-4c).


In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Text;

// Separateur decimal invariant (CultureInfo) -> sorties reproductibles.
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

//Nombre de marches (entrants sequentiels).
const int N = 5;

var sb = new StringBuilder();
sb.AppendLine($"Chaine de Magasins - Information Complete ({N} marches)");
sb.AppendLine(new string('=', 60));
sb.AppendLine();
sb.AppendLine("Analyse par induction arriere:");
sb.AppendLine(new string('-', 40));
sb.AppendLine($"  Marche {N} (dernier):");
sb.AppendLine("    Si entree, Monopole: Accommodate (1 > -1)");
sb.AppendLine("    Entrant: Enter (1 > 0)");
sb.AppendLine();
sb.AppendLine($"  Marche {N - 1}:");
sb.AppendLine("    Meme raisonnement: Entrant entre, Monopole accommode");
sb.AppendLine("    Car 'Fight' ne change rien au marche suivant !");
sb.AppendLine();
sb.AppendLine("  ... (recursivement)");
sb.AppendLine();
sb.AppendLine("  Marche 1:");
sb.AppendLine("    Entrant entre, Monopole accommode");
sb.AppendLine();
sb.AppendLine(new string('=', 60));
sb.AppendLine();
sb.AppendLine("Resultat a l'equilibre:");
int totalMonopole = N * 1;
int totalEntrants = N * 1;
sb.AppendLine("  Tous les entrants entrent");
sb.AppendLine("  Monopole accommode toujours");
sb.AppendLine($"  Gain Monopole: {totalMonopole}");
sb.AppendLine($"  Gain total Entrants: {totalEntrants}");
sb.AppendLine();
sb.AppendLine(new string('=', 60));
sb.AppendLine();
sb.AppendLine("Paradoxe:");
sb.AppendLine("  Si le monopole pouvait CREDIBLEMENT menacer de combattre,");
sb.AppendLine($"  il pourrait dissuader l'entree et gagner {N * 2} !");
sb.Append("  Mais l'induction arriere rend cette menace non-credible.");

sb.ToString().Display();


The below script needs to be able to find the current output cell; this is an easy method to get it.

Chaine de Magasins - Information Complete (5 marches)

Analyse par induction arriere:
----------------------------------------
  Marche 5 (dernier):
    Si entree, Monopole: Accommodate (1 > -1)
    Entrant: Enter (1 > 0)

  Marche 4:
    Meme raisonnement: Entrant entre, Monopole accommode
    Car 'Fight' ne change rien au marche suivant !

  ... (recursivement)

  Marche 1:
    Entrant entre, Monopole accommode


Resultat a l'equilibre:
  Tous les entrants entrent
  Monopole accommode toujours
  Gain Monopole: 5
  Gain total Entrants: 5


Paradoxe:
  Si le monopole pouvait CREDIBLEMENT menacer de combattre,
  il pourrait dissuader l'entree et gagner 10 !
  Mais l'induction arriere rend cette menace non-credible.

## Information incomplète : pourquoi la réputation change tout

En information complète, l'induction arrière "defait" le paradoxe (menace non-crédible). L'intuition de
**Kreps-Wilson** : s'il existe une *toute petite* probabilité $\epsilon$ que l'incumbent soit irrationnel
(un type "tough" qui combat toujours), alors un incumbent rationnel a intérêt a **imiter** le type tough
les premiers marches pour bâtir une réputation de combattant, et ainsi dissuader les entrants suivants.
La menace devient crédible *grace a l'incertitude sur le type*.

Cela mobilise deux outils : le **cheap talk** (communication non couteuse, Crawford-Sobel) et la **mise à jour
bayésienne des croyances** (Kreps-Wilson, KMRW).


In [2]:
#nullable enable
using System;
using System.Globalization;
using System.Text;

// ---- Analyse qualitative du cheap talk ----
var intro = new StringBuilder();
intro.AppendLine("Cheap Talk - Communication Non-Couteuse");
intro.AppendLine(new string('=', 60));
intro.AppendLine();
intro.AppendLine("1. Exemple: Embauche");
intro.AppendLine(new string('-', 40));
intro.AppendLine("  Candidat: 'Je suis tres motive !'");
intro.AppendLine("  Employeur: '...'");
intro.AppendLine();
intro.AppendLine("  -> Non-informatif: TOUS les candidats disent ca !");
intro.AppendLine("     (Memes les non-motives ont interet a mentir)");
intro.AppendLine();
intro.AppendLine("2. Quand le cheap talk fonctionne:");
intro.AppendLine(new string('-', 40));
intro.AppendLine("  a) Interets alignes : pas d'incitation a mentir.");
intro.AppendLine("  b) Interets partiellement alignes : equilibres avec 'partition' des types.");
intro.AppendLine("  c) Verification ex-post : mensonge detectable + penalite de reputacion.");
intro.AppendLine();
intro.AppendLine("3. Modele Crawford-Sobel (simplifie):");
intro.AppendLine(new string('-', 40));
intro.AppendLine("  - Expert connait theta dans [0,1], envoie message m.");
intro.AppendLine("  - Decideur choisit action a.");
intro.AppendLine("  - Expert: -(a - theta - b)^2  (biais b > 0)");
intro.AppendLine("  - Decideur: -(a - theta)^2");
intro.AppendLine();
intro.Append("  Resultat: plus b est grand, moins d'information transmise ; babbling si b trop grand.");
intro.ToString().Display();

// ---- Crawford-Sobel : equilibre de partition + information transmise ----
// Reproduction fidele de l'heuristique Python : a_i = i/n + 2*b*i*(n-i).
(double bias, double varTotal) = (0.0, 1.0 / 12.0);
var sweep = new StringBuilder();
sweep.AppendLine();
sweep.AppendLine("Sweep du biais (formule de partition a_i = i/n + 2*b*i*(n-i))");
sweep.AppendLine(new string('-', 60));

foreach (double b in new[] { 0.05, 0.1, 0.2, 0.3 })
{
    int nMax = b < 0.25 ? (int)Math.Floor(1.0 / (4.0 * b) + 0.5) : 1;
    int n = nMax;
    // Frontieres a_i
    var boundaries = new double[n + 1];
    for (int i = 0; i <= n; i++)
        boundaries[i] = (double)i / n + 2.0 * b * i * (n - i);
    // Variance intra-partition (ponderation prob = largeur)
    double varIntra = 0.0;
    for (int i = 0; i < n; i++)
    {
        double width = boundaries[i + 1] - boundaries[i];
        double prob = width;
        double varPartition = width * width / 12.0;
        varIntra += prob * varPartition;
    }
    double infoTransmitted = 1.0 - varIntra / varTotal;

    var bndStr = new StringBuilder();
    for (int i = 0; i <= n; i++)
    {
        if (i > 0) bndStr.Append(", ");
        bndStr.Append(Math.Round(boundaries[i], 3).ToString(CultureInfo.InvariantCulture));
    }
    sweep.AppendLine($"  b={b,4:F2}  n_max={nMax}  frontieres=[{bndStr}]  info={infoTransmitted * 100,5:F1}%");
}
sweep.AppendLine();
sweep.Append("Lecture: b=0.20 et 0.30 -> equilibre babbling (1 partition, 0% d'info). "
           + "A petit biais, plusieurs partitions ; noter que la formule simplifiee devient non-monotone "
           + "quand b depasse le seuil de validite b <= 1/(2*n*(n-1)) (limitation connue de l'approximation).");
sweep.ToString().Display();


Cheap Talk - Communication Non-Couteuse

1. Exemple: Embauche
----------------------------------------
  Candidat: 'Je suis tres motive !'
  Employeur: '...'

  -> Non-informatif: TOUS les candidats disent ca !
     (Memes les non-motives ont interet a mentir)

2. Quand le cheap talk fonctionne:
----------------------------------------
  a) Interets alignes : pas d'incitation a mentir.
  b) Interets partiellement alignes : equilibres avec 'partition' des types.
  c) Verification ex-post : mensonge detectable + penalite de reputacion.

3. Modele Crawford-Sobel (simplifie):
----------------------------------------
  - Expert connait theta dans [0,1], envoie message m.
  - Decideur choisit action a.
  - Expert: -(a - theta - b)^2  (biais b > 0)
  - Decideur: -(a - theta)^2

  Resultat: plus b est grand, moins d'information transmise ; babbling si b trop grand.


Sweep du biais (formule de partition a_i = i/n + 2*b*i*(n-i))
------------------------------------------------------------
  b=0.05  n_max=5  frontieres=[0, 0.6, 1, 1.2, 1.2, 1]  info= 72.0%
  b=0.10  n_max=3  frontieres=[0, 0.733, 1.067, 1]  info= 56.9%
  b=0.20  n_max=1  frontieres=[0, 1]  info=  0.0%
  b=0.30  n_max=1  frontieres=[0, 1]  info=  0.0%

Lecture: b=0.20 et 0.30 -> equilibre babbling (1 partition, 0% d'info). A petit biais, plusieurs partitions ; noter que la formule simplifiee devient non-monotone quand b depasse le seuil de validite b <= 1/(2*n*(n-1)) (limitation connue de l'approximation).

## Modèle de Kreps-Wilson : réputation du "tough incumbent"

**Paramètres** :
- $n$ marches (entrants séquentiels).
- $\epsilon$ = probabilité à priori que l'incumbent soit de type **Fou** (combat toujours, prefere Fight).
- type **Normal** : Fight=-1, Accommodate=1, pas d'entree=2.
- L'entrant observe l'historique des combats et met à jour sa croyance $P(\text{Fou} \mid h)$ par la règle de Bayes.

**Seuil de dissuasion** : un entrant rationnel entre si son gain espere est positif.
$E[\text{gain}] = P(\text{Fou}) \cdot (-1) + (1 - P(\text{Fou})) \cdot 1 = 0 \Rightarrow P(\text{Fou}) = 0.5$.
Des que la croyance depasse 0.5, l'entrant reste dehors.


In [3]:
#nullable enable
using System;
using System.Globalization;
using System.Text;

const int N = 10;
// Non-const : evite le pliage par le compilateur (CS0162 code inaccessible) et garde la
// logique conditionnelle lisible (les deux branches sont pedagogiquement pertinentes).
double Epsilon = 0.1;
double Seuil = 0.5;

var sb = new StringBuilder();
sb.AppendLine($"Kreps-Wilson: Chaine de Magasins avec Reputation");
sb.AppendLine(new string('=', 60));
sb.AppendLine($"Nombre de marches: {N}");
sb.AppendLine($"Probabilite du type Fou: epsilon = {Epsilon}");
sb.AppendLine();
sb.AppendLine("Gains:");
sb.AppendLine("  Type Normal: Fight=-1, Accommodate=1, No entry=2");
sb.AppendLine("  Type Fou: Fight=1 (toujours), No entry=2");
sb.AppendLine("  Entrant: Enter+Accommodate=1, Enter+Fight=-1, Out=0");
sb.AppendLine();
sb.AppendLine(new string('=', 60));
sb.AppendLine();
sb.AppendLine("Equilibre (intuitif):");
sb.AppendLine();
sb.AppendLine($"Seuil de reputation: {Seuil}");
sb.AppendLine($"  Si P(Fou) >= {Seuil}, l'entrant n'entre pas");
sb.AppendLine();
sb.AppendLine(new string('=', 60));
sb.AppendLine();
sb.AppendLine("Dynamique de reputation:");
sb.AppendLine();
sb.AppendLine($"  Croyance initiale: P(Fou) = {Epsilon:F3}");
sb.AppendLine();
sb.AppendLine("  Strategies possibles du monopole Normal:");
sb.AppendLine("    - Pooling: toujours Fight (imite le Fou)");
sb.AppendLine("    - Separateur: toujours Accommodate (revele son type)");
sb.AppendLine("    - Mixte: Fight avec probabilite pour maintenir reputation");
sb.AppendLine();
sb.AppendLine(new string('=', 60));
sb.AppendLine();
sb.AppendLine("Resultat qualitatif:");

if (Epsilon >= Seuil)
{
    sb.AppendLine($"  epsilon = {Epsilon} >= {Seuil}");
    sb.AppendLine($"  -> Aucun entrant n'entre ! Monopole gagne 2*{N} = {2 * N}");
}
else
{
    // Approximation Python : k_dissuaded = floor(log2(seuil/epsilon))
    int kDissuaded = Math.Max(0, (int)Math.Floor(Math.Log(Seuil / Epsilon) / Math.Log(2)));
    kDissuaded = Math.Min(kDissuaded, N - 1);
    sb.AppendLine($"  epsilon = {Epsilon} < {Seuil}");
    sb.AppendLine($"  -> Environ {kDissuaded} premiers entrants dissuades");
    sb.Append("  -> Monopole gagne plus qu'avec info complete !");
}
sb.ToString().Display();


Kreps-Wilson: Chaine de Magasins avec Reputation
Nombre de marches: 10
Probabilite du type Fou: epsilon = 0.1

Gains:
  Type Normal: Fight=-1, Accommodate=1, No entry=2
  Type Fou: Fight=1 (toujours), No entry=2
  Entrant: Enter+Accommodate=1, Enter+Fight=-1, Out=0


Equilibre (intuitif):

Seuil de reputation: 0.5
  Si P(Fou) >= 0.5, l'entrant n'entre pas


Dynamique de reputation:

  Croyance initiale: P(Fou) = 0.100

  Strategies possibles du monopole Normal:
    - Pooling: toujours Fight (imite le Fou)
    - Separateur: toujours Accommodate (revele son type)
    - Mixte: Fight avec probabilite pour maintenir reputation


Resultat qualitatif:
  epsilon = 0.1 < 0.5
  -> Environ 2 premiers entrants dissuades
  -> Monopole gagne plus qu'avec info complete !

## Simulation numérique Monte-Carlo

Stratégie simplifiée pour la simulation :
- **Fou** : Fight systématiquement.
- **Normal** : combat tant qu'il reste des marches et que la réputation est sous le seuil (bâtir la réputation), accommode sinon.
- **Entrant** : entre si $P(\text{Fou}) < 0.5$, sinon reste hors.
- **Mise à jour bayésienne** après un combat observe : $P(\text{Fou} \mid F) = \frac{P(F)}{P(F) + (1-P(F)) \cdot p^{\text{fight}}_{\text{normal}}}$.

Le PRNG est `System.Random` seede (reproductible). Les valeurs numériques dependent du PRNG ;
l'**écart qualitatif** (le Normal investit des combats précoces pour gagner ensuite) est la leçon.


In [4]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;

static (double meanNormal, double meanFou, double meanEntrants, int nNormal, int nFou)
    SimulateReputationGame(int nMarkets, double epsilon, int nSimulations, int seed)
{
    var rng = new Random(seed);
    const double Seuil = 0.5;
    var gainsNormal = new List<double>();
    var gainsFou = new List<double>();
    var gainsEntrants = new List<double>();

    for (int s = 0; s < nSimulations; s++)
    {
        bool isFou = rng.NextDouble() < epsilon;
        double reputation = epsilon;
        double gainMonopole = 0;
        double gainEntrantsTotal = 0;
        int nFights = 0;

        for (int market = 0; market < nMarkets; market++)
        {
            if (reputation >= Seuil)
            {
                gainMonopole += 2;
                gainEntrantsTotal += 0;
            }
            else
            {
                if (isFou)
                {
                    gainMonopole += 1;
                    gainEntrantsTotal += -1;
                    nFights++;
                }
                else
                {
                    int remaining = nMarkets - market - 1;
                    if (remaining > 0 && reputation < Seuil)
                    {
                        gainMonopole -= 1;
                        gainEntrantsTotal += -1;
                        nFights++;
                    }
                    else
                    {
                        gainMonopole += 1;
                        gainEntrantsTotal += 1;
                    }
                }
                if (nFights > 0)
                {
                    double pFightNormal = (market < nMarkets - 2) ? 0.8 : 0.1;
                    reputation = reputation / (reputation + (1.0 - reputation) * pFightNormal);
                }
            }
        }

        if (isFou) gainsFou.Add(gainMonopole); else gainsNormal.Add(gainMonopole);
        gainsEntrants.Add(gainEntrantsTotal);
    }

    return (
        gainsNormal.Count > 0 ? gainsNormal.Average() : double.NaN,
        gainsFou.Count > 0 ? gainsFou.Average() : double.NaN,
        gainsEntrants.Average(),
        gainsNormal.Count,
        gainsFou.Count);
}

var (meanNormal, meanFou, meanEntrants, nNormal, nFou) = SimulateReputationGame(10, 0.1, 10000, seed: 42);

var sb = new StringBuilder();
sb.AppendLine($"Simulation Monte-Carlo: 10 marches, epsilon=0.1, 10000 sims (seed=42)");
sb.AppendLine(new string('-', 60));
sb.AppendLine($"  Effectifs   : Normal n={nNormal}, Fou n={nFou}");
sb.AppendLine($"  Monopole Normal - Gain moyen : {meanNormal,7:F2}");
sb.AppendLine($"  Monopole Fou     - Gain moyen : {meanFou,7:F2}");
sb.AppendLine($"  Entrants (total) - Gain moyen : {meanEntrants,7:F2}");
sb.AppendLine();
sb.AppendLine("  Reference (info complete): Monopole = 10, Entrants = 10");
sb.Append("  -> Le type Normal investit (-1) sur les combats precoces pour batir la reputacion.");
sb.ToString().Display();


Simulation Monte-Carlo: 10 marches, epsilon=0.1, 10000 sims (seed=42)
------------------------------------------------------------
  Effectifs   : Normal n=8959, Fou n=1041
  Monopole Normal - Gain moyen :   -7.00
  Monopole Fou     - Gain moyen :   11.00
  Entrants (total) - Gain moyen :   -9.00

  Reference (info complete): Monopole = 10, Entrants = 10
  -> Le type Normal investit (-1) sur les combats precoces pour batir la reputacion.

## Impact du paramètre $\epsilon$ sur les gains

Variation de la probabilité du type Fou : à mesure qu'$\epsilon$ croît, l'entrant est plus dissuade,
le monopole Normal bénéficie d'une réputation plus forte. Visualisation en table console (équivalent ASCII
du plot matplotlib du notebook Python).


In [5]:
#nullable enable
using System;
using System.Globalization;
using System.Text;

var epsilons = new[] { 0.01, 0.05, 0.1, 0.2, 0.3, 0.5 };
const int N = 10;
var sb = new StringBuilder();
sb.AppendLine("Impact de epsilon sur les gains (n_markets=10, 5000 sims/point)");
sb.AppendLine(new string('=', 70));
sb.AppendLine("  " + "eps".PadRight(6) + "  " + "Monopole Normal".PadRight(16) + "  " + "Entrants(total)".PadRight(16) + "  " + "ref info complete".PadRight(18));
sb.AppendLine(new string(' ', 8) + new string('-', 16) + "  " + new string('-', 16) + "  " + new string('-', 18));

// Reproductibilite : meme seed relance pour chaque point (chaque point est sa propre simulation).
foreach (double eps in epsilons)
{
    // meme moteur de simulation que la cellule precedente (recopie minimale pour l'autonomie de la cellule).
    var rng = new Random(123);
    const double Seuil = 0.5;
    var gainsNormal = new System.Collections.Generic.List<double>();
    var gainsEntrants = new System.Collections.Generic.List<double>();
    int sims = 5000;
    for (int s = 0; s < sims; s++)
    {
        bool isFou = rng.NextDouble() < eps;
        double reputation = eps;
        double gm = 0; double ge = 0; int nf = 0;
        for (int market = 0; market < N; market++)
        {
            if (reputation >= Seuil) { gm += 2; ge += 0; }
            else
            {
                if (isFou) { gm += 1; ge += -1; nf++; }
                else
                {
                    int remaining = N - market - 1;
                    if (remaining > 0 && reputation < Seuil) { gm -= 1; ge += -1; nf++; }
                    else { gm += 1; ge += 1; }
                }
                if (nf > 0)
                {
                    double pfn = (market < N - 2) ? 0.8 : 0.1;
                    reputation = reputation / (reputation + (1.0 - reputation) * pfn);
                }
            }
        }
        if (!isFou) gainsNormal.Add(gm);
        gainsEntrants.Add(ge);
    }
    double meanN = gainsNormal.Count > 0 ? MeanD(gainsNormal) : double.NaN;
    double meanE = MeanD(gainsEntrants);
    sb.AppendLine($"  {eps,6:F2}  {meanN,16:F2}  {meanE,16:F2}  {N,18}");
}
sb.AppendLine();
sb.AppendLine("  Repere ideal Monopole (dissuasion totale) = 2*n = 20.");
sb.ToString().Display();

static double MeanD(System.Collections.Generic.List<double> xs) { double s = 0; foreach (var x in xs) s += x; return s / xs.Count; }


Impact de epsilon sur les gains (n_markets=10, 5000 sims/point)
  eps     Monopole Normal   Entrants(total)   ref info complete 
        ----------------  ----------------  ------------------
    0.01             -8.00             -8.02                  10
    0.05             -7.00             -9.00                  10
    0.10             -7.00             -9.00                  10
    0.20             -1.00             -7.00                  10
    0.30              8.00             -4.00                  10
    0.50             20.00              0.00                  10

  Repere ideal Monopole (dissuasion totale) = 2*n = 20.


## KMRW : coopération dans le dilemme du prisonnier fini

Selten (1978) a montre par induction arrière que le dilemme du prisonnier répété un nombre *fini* de fois
se solde par la défection systématique. **KMRW (1982)** renversent ce résultat : s'il existe une petite
probabilité $\epsilon$ que l'adversaire joue **Tit-for-Tat** (TFT), alors la coopération devient rationnelle
pendant la majeure partie du jeu, même fini. "A small amount of incomplète information can have a large effect
on the equilibrium of a game."


In [6]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Text;

const int T = 20;
const double Epsilon = 0.05;

var sb = new StringBuilder();
sb.AppendLine($"KMRW: Dilemme du Prisonnier Repete {T} fois");
sb.AppendLine(new string('=', 60));
sb.AppendLine($"Probabilite du type TFT: epsilon = {Epsilon}");
sb.AppendLine();
sb.AppendLine("Matrice de gains:");
sb.AppendLine("         C       D");
sb.AppendLine("  C    (3,3)   (0,5)");
sb.AppendLine("  D    (5,0)   (1,1)");
sb.AppendLine();
sb.AppendLine(new string('=', 60));
sb.AppendLine();
sb.AppendLine("Analyse:");

int gainDefectAll = T * 1;
sb.AppendLine();
sb.AppendLine("  Sans incertitude (info complete):");
sb.AppendLine("    Equilibre: (D, D) a chaque tour");
sb.AppendLine($"    Gain par joueur: {gainDefectAll}");

sb.AppendLine();
sb.AppendLine($"  Avec incertitude (epsilon = {Epsilon}):");

// Heuristique Python : cooperation_rounds = T - floor(log2(1/epsilon))
int cooperationRounds = Math.Max(0, T - (int)Math.Floor(Math.Log(1.0 / Epsilon) / Math.Log(2)));
int gainCoop = cooperationRounds * 3 + (T - cooperationRounds) * 1;
sb.AppendLine($"    Cooperation pendant environ {cooperationRounds} tours (heuristique)");
sb.AppendLine($"    Gain approximatif par joueur: {gainCoop}");
sb.AppendLine();
sb.Append($"    Amelioration: {gainCoop - gainDefectAll} ({(gainCoop / (double)gainDefectAll - 1) * 100:F1}%)");
sb.ToString().Display();

// ---- Simulation PD repete avec types (TFT vs heuristique Normal) ----
static (double meanP1, double meanP2, double[] coopByRound) SimulatePdWithReputation(int T, double epsilon, int nSims, int seed)
{
    var rng = new Random(seed);
    double p1 = 0, p2 = 0;
    var coop = new double[T];
    for (int s = 0; s < nSims; s++)
    {
        bool p1Tft = rng.NextDouble() < epsilon;
        bool p2Tft = rng.NextDouble() < epsilon;
        var h1 = new List<char>(); // actions de P1
        var h2 = new List<char>(); // actions de P2
        int g1 = 0, g2 = 0;
        for (int t = 0; t < T; t++)
        {
            int remaining = T - t;
            char a1, a2;
            // P1
            if (p1Tft) a1 = (h2.Count == 0) ? 'C' : h2[h2.Count - 1];
            else a1 = (remaining > 3 && (h2.Count == 0 || h2[h2.Count - 1] == 'C')) ? 'C' : 'D';
            // P2
            if (p2Tft) a2 = (h1.Count == 0) ? 'C' : h1[h1.Count - 1];
            else a2 = (remaining > 3 && (h1.Count == 0 || h1[h1.Count - 1] == 'C')) ? 'C' : 'D';
            // gains
            int r1, r2;
            if (a1 == 'C' && a2 == 'C') { r1 = 3; r2 = 3; }
            else if (a1 == 'C' && a2 == 'D') { r1 = 0; r2 = 5; }
            else if (a1 == 'D' && a2 == 'C') { r1 = 5; r2 = 0; }
            else { r1 = 1; r2 = 1; }
            g1 += r1; g2 += r2;
            if (a1 == 'C' && a2 == 'C') coop[t]++;
            h1.Add(a1); h2.Add(a2);
        }
        p1 += g1; p2 += g2;
    }
    for (int t = 0; t < T; t++) coop[t] /= nSims;
    return (p1 / nSims, p2 / nSims, coop);
}

var (m1, m2, coopRate) = SimulatePdWithReputation(T, Epsilon, nSims: 5000, seed: 7);
var sb2 = new StringBuilder();
sb2.AppendLine();
sb2.AppendLine($"Simulation PD repete (T={T}, eps={Epsilon}, 5000 sims, seed=7)");
sb2.AppendLine(new string('-', 60));
sb2.AppendLine($"  Gain moyen P1 : {m1:F2}");
sb2.AppendLine($"  Gain moyen P2 : {m2:F2}");
sb2.AppendLine($"  Gain defection pure (ref) : {T}");
sb2.AppendLine();
sb2.AppendLine("  Taux de cooperation mutuelle (C,C) par bloc de tours:");
for (int t = 0; t < T; t += 4)
{
    int end = Math.Min(t + 4, T);
    double avg = 0; int cnt = 0;
    for (int i = t; i < end; i++) { avg += coopRate[i]; cnt++; }
    avg /= cnt;
    string bar = new string('#', (int)Math.Round(avg * 20));
    sb2.AppendLine($"    tours {t + 1,2}-{end,2}: {avg * 100,5:F1}%  {bar}");
}
sb2.AppendLine();
sb2.Append("  -> Cooperation elevee en debut de jeu, effondrement sur la fin (effet horizon fini).");
sb2.ToString().Display();


KMRW: Dilemme du Prisonnier Repete 20 fois
Probabilite du type TFT: epsilon = 0.05

Matrice de gains:
         C       D
  C    (3,3)   (0,5)
  D    (5,0)   (1,1)


Analyse:

  Sans incertitude (info complete):
    Equilibre: (D, D) a chaque tour
    Gain par joueur: 20

  Avec incertitude (epsilon = 0.05):
    Cooperation pendant environ 16 tours (heuristique)
    Gain approximatif par joueur: 52

    Amelioration: 32 (160.0%)


Simulation PD repete (T=20, eps=0.05, 5000 sims, seed=7)
------------------------------------------------------------
  Gain moyen P1 : 54.15
  Gain moyen P2 : 54.17
  Gain defection pure (ref) : 20

  Taux de cooperation mutuelle (C,C) par bloc de tours:
    tours  1- 4: 100.0%  ####################
    tours  5- 8: 100.0%  ####################
    tours  9-12: 100.0%  ####################
    tours 13-16: 100.0%  ####################
    tours 17-20:  25.2%  #####

  -> Cooperation elevee en debut de jeu, effondrement sur la fin (effet horizon fini).

## Équilibre bayésien parfait (PBE)

Un PBE precise, pour chaque ensemble d'information :
1. **Un profil de stratégies** $\sigma_i(t_i, h)$ dépendant du type et de l'historique.
2. **Un système de croyances** $\mu(t \mid h)$ dérivé par la règle de Bayes quand c'est possible.

Il doit satisfaire **consistance** (Bayes sur le chemin d'équilibre) et **rationalité** (chaque stratégie est
optimale étant donne les croyances). Les équilibres de signaling se classent en **séparateur** (types différents
-> signaux différents, le type est révèle) et **pooling** (tous les types envoient le même signal).


In [7]:
#nullable enable
using System.Text;

var sb = new StringBuilder();
sb.AppendLine("Equilibre Bayesien Parfait (PBE)");
sb.AppendLine(new string('=', 60));
sb.AppendLine();
sb.AppendLine("1. Composantes:");
sb.AppendLine(new string('-', 40));
sb.AppendLine("  a) Profil de strategies: sigma_i(t_i, h) pour chaque joueur");
sb.AppendLine("     - Depend du type t_i et de l'historique h");
sb.AppendLine();
sb.AppendLine("  b) Systeme de croyances: mu(t | h) pour chaque info set");
sb.AppendLine("     - Probabilite des types des autres conditionnelle a h");
sb.AppendLine();
sb.AppendLine("2. Conditions d'equilibre:");
sb.AppendLine(new string('-', 40));
sb.AppendLine("  a) Consistance: mu derive de sigma par Bayes quand possible");
sb.AppendLine("     mu(t | h) = P(h | t) * P(t) / P(h)");
sb.AppendLine();
sb.AppendLine("  b) Rationalite: sigma optimal etant donne mu");
sb.AppendLine("     sigma_i(t_i, h) in argmax E[u_i | t_i, h, mu]");
sb.AppendLine();
sb.AppendLine("3. Exemple: Jeu de signaling");
sb.AppendLine(new string('-', 40));
sb.AppendLine("  Emetteur (S): Nature tire type in {H, L}");
sb.AppendLine("  S observe type, envoie signal m in {m1, m2}");
sb.AppendLine("  Recepteur (R): observe m, choisit action a");
sb.AppendLine();
sb.AppendLine("  PBE specifie:");
sb.AppendLine("    - sigma_S: strategie de S pour chaque type");
sb.AppendLine("    - sigma_R: strategie de R pour chaque signal");
sb.AppendLine("    - mu: croyance de R sur le type apres chaque signal");
sb.AppendLine();
sb.AppendLine("4. Types d'equilibres de signaling:");
sb.AppendLine(new string('-', 40));
sb.AppendLine("  - Separateur: types differents -> signaux differents");
sb.AppendLine("    mu revele parfaitement le type");
sb.AppendLine("  - Pooling: tous les types -> meme signal");
sb.Append("    croyance posterieure = prior (aucune information revelee)");
sb.ToString().Display();


Equilibre Bayesien Parfait (PBE)

1. Composantes:
----------------------------------------
  a) Profil de strategies: sigma_i(t_i, h) pour chaque joueur
     - Depend du type t_i et de l'historique h

  b) Systeme de croyances: mu(t | h) pour chaque info set
     - Probabilite des types des autres conditionnelle a h

2. Conditions d'equilibre:
----------------------------------------
  a) Consistance: mu derive de sigma par Bayes quand possible
     mu(t | h) = P(h | t) * P(t) / P(h)

  b) Rationalite: sigma optimal etant donne mu
     sigma_i(t_i, h) in argmax E[u_i | t_i, h, mu]

3. Exemple: Jeu de signaling
----------------------------------------
  Emetteur (S): Nature tire type in {H, L}
  S observe type, envoie signal m in {m1, m2}
  Recepteur (R): observe m, choisit action a

  PBE specifie:
    - sigma_S: strategie de S pour chaque type
    - sigma_R: strategie de R pour chaque signal
    - mu: croyance de R sur le type apres chaque signal

4. Types d'equilibres de signaling:
-

## Exercices

Les trois exercices suivants sont à compléter (stubs sans erreur, le notebook s'execute de bout en bout).
Conventions : `pass`/`return null`/`print("Exercice à compléter")` — jamais de `throw` intentionnel.


In [8]:
#nullable enable
using System.Text;

// Exercice 1 : Beer-Quiche (Cho-Kreps 1987)
// Indice : P(Strong)=0.9, P(Weak)=0.1 ; Strong prefere Beer, Weak prefere Quiche.
// Le Receiver choisit Fight ou Not apres avoir observe le signal.
// Etapes : 1) types + probabilites  2) strategies par type
//          3) PBE (pooling Beer ? pooling Quiche ? separateur ?)  4) croyances hors-equilibre.
string SolveBeerQuiche()
{
    // TODO etudiant : analyser le jeu Beer-Quiche et retourner une chaine decrivant le(s) PBE.
    return "Exercice a completer";
}

SolveBeerQuiche().Display();


Exercice a completer

In [9]:
#nullable enable
using System;
using System.Collections.Generic;

// Exercice 2 : Chain Store Paradox avec reputacion.
// Parametres : N_ENTRANTS=20, P_HARD=0.3 (proba a priori que l'incumbent soit "dur").
//   payoffs : hard    {Fight: 0,  Accommodate: -1}
//             rational{Fight: -2, Accommodate:  1}
//             entrant {Enter_if_fight: -2, Enter_if_accommodate: 2, Stay: 0}
//
// Etape 1 : implementer la mise a jour des croyances (Bayes) apres observation d'une action.
//   P(hard | Fight) = P(Fight | hard) * P(hard) / P(Fight)
// Etape 2 : simuler le jeu avec mise a jour des croyances.
// Etape 3 : analyser l'effet de la reputacion (sweep p in {0.1, 0.3, 0.5}).

double UpdateBelief(double prior, string action)
{
    // TODO etudiant : retourner P(hard | action) par la regle de Bayes.
    return 0.0;  // TODO etudiant
}

Dictionary<string, object> SimulateChainStore(double pHard, int nEntrants)
{
    // TODO etudiant : simuler la chaine de magasins avec mise a jour bayesienne.
    return new Dictionary<string, object> { ["fights"] = 0, ["accommodations"] = 0 };  // TODO etudiant
}

Console.WriteLine("Exercice a completer");


Exercice a completer


In [10]:
#nullable enable
using System;
using System.Collections.Generic;

// Exercice 3 : Dilemme du Prisonnier repete — TFT vs best-response myope sur 10 tours.
// Etape 1 : definir les deux strategies (TFT : coopere puis copie ; myope : D si l'autre a D, C sinon).
// Etape 2 : simulation sur 10 tours avec une matrice de gains standard.
// Etape 3 : comparaison des gains cumules.

Dictionary<string, double> SimulateTftVsMyopic(Dictionary<(char,char),(double,double)> payoff, int nRounds = 10)
{
    // TODO etudiant : implementer la simulation TFT vs myope.
    return new Dictionary<string, double>
    {
        ["tft_payoff"] = 0.0,
        ["myopic_payoff"] = 0.0,
    };  // TODO etudiant
}

Console.WriteLine("Exercice a completer");


Exercice a completer


## Conclusion

Les jeux de réputation montrent qu'une **toute petite incertitude sur le type** d'un joueur change
radicalement l'équilibre :

- **Chain Store Paradox** : menace non-crédible en information complète → crédible dès qu'on introduit $\epsilon$.
- **KMRW** : la coopération émerge dans le dilemme du prisonnier fini malgré l'induction arrière.
- **Crawford-Sobel** : même communication gratuite, l'information transmise décroît avec le biais.

L'outil central est la **mise à jour bayésienne des croyances** : chaque action observée est un signal qui
façonne la réputation, et cette réputation contraint les décisions futures de tous les joueurs. C'est le
pont entre information incomplète et équilibre (PBE). Ce twin C# reproduit from-scratch, sans aucune
librairie externe, les algorithmes du notebook Python.
